In [0]:
create database if not exists sales_scd;

In [0]:
use sales_scd;

CREATE TABLE IF NOT EXISTS sales_scd.orders (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(100),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10, 2),
    TotalAmount DECIMAL(10, 2)
  );

In [0]:
INSERT INTO sales_scd.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00),
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00),
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00),
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00),
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);

num_affected_rows,num_inserted_rows
10,10


## SCD TYPE-1

In [0]:
select * from sales_scd.orders

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,david@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


In [0]:
create or replace view sales_scd.view_DimProducts
AS
select distinct(ProductID) as ProductID,ProductName, ProductCategory from sales_scd.orders
where OrderDate > '2024-02-10'

In [0]:
CREATE OR REPLACE TABLE sales_scd.DimProducts
(
    ProductID INT,
    ProductName STRING,
    ProductCategory STRING
)

In [0]:
INSERT INTO sales_scd.DimProducts
SELECT ProductID, ProductName, ProductCategory FROM sales_scd.view_dimproducts

num_affected_rows,num_inserted_rows
9,9


In [0]:
SELECT * FROM sales_scd.DimProducts

ProductID,ProductName,ProductCategory
204,Headphones,Accessories
205,Gaming Console,Electronics
209,Mouse,Accessories
202,Smartphone,Electronics
206,Smartwatch,Electronics
201,Laptop,Electronics
203,Tablet,Electronics
207,Monitor,Accessories
208,Keyboard,Accessories


In [0]:
INSERT INTO sales_scd.orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount)
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Gaming Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-12', 102, 'Bob Smith', 'bob@example.com', 230, 'Airpods', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00)

num_affected_rows,num_inserted_rows
2,2


In [0]:
SELECT * FROM sales_scd.view_dimproducts

ProductID,ProductName,ProductCategory
230,Airpods,Electronics


## Merge - SCD Type 1

In [0]:
MERGE INTO sales_scd.DimProducts AS trg
USING sales_scd.view_DimProducts AS src
ON trg.ProductID = src.ProductID
WHEN MATCHED THEN
UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
1,0,0,1


In [0]:
SELECT * FROM sales_scd.dimproducts

ProductID,ProductName,ProductCategory
204,Headphones,Accessories
205,Gaming Console,Electronics
209,Mouse,Accessories
202,Smartphone,Electronics
206,Smartwatch,Electronics
201,Laptop,Electronics
203,Tablet,Electronics
207,Monitor,Accessories
208,Keyboard,Accessories
230,Airpods,Electronics


## SCD TYPE-2

In [0]:
CREATE OR REPLACE TABLE sales_scd.DimProducts_SCD2
(
    ProductKey INT,
    ProductID INT,
    ProductName STRING,
    ProductCategory STRING,
    StartDate DATE,
    EndDate DATE,
    IsCurrent BOOLEAN
);

In [0]:
INSERT INTO sales_scd.DimProducts_SCD2
SELECT
    ROW_NUMBER() OVER (ORDER BY ProductID) AS ProductKey,
    ProductID,
    ProductName,
    ProductCategory,
    OrderDate AS StartDate,
    NULL AS EndDate,
    TRUE AS IsCurrent
FROM sales_scd.orders
WHERE OrderDate <= '2024-02-10';

num_affected_rows,num_inserted_rows
21,21


In [0]:
UPDATE sales_scd.DimProducts_SCD2
SET
    EndDate = '2024-02-11',
    IsCurrent = FALSE
WHERE ProductID = 201
  AND IsCurrent = TRUE;

num_affected_rows
5


In [0]:
INSERT INTO sales_scd.DimProducts_SCD2
VALUES
(
    2,
    201,
    'Gaming Laptop',
    'Electronics',
    '2024-02-12',
    NULL,
    TRUE
);

num_affected_rows,num_inserted_rows
1,1


In [0]:
SELECT * FROM sales_scd.dimproducts_scd2

ProductKey,ProductID,ProductName,ProductCategory,StartDate,EndDate,IsCurrent
6,202,Smartphone,Electronics,2024-02-02,null,true
7,202,Smartphone,Electronics,2024-02-02,null,true
8,203,Tablet,Electronics,2024-02-03,null,true
9,203,Tablet,Electronics,2024-02-03,null,true
10,204,Headphones,Accessories,2024-02-04,null,true
11,204,Headphones,Accessories,2024-02-04,null,true
12,205,Gaming Console,Electronics,2024-02-05,null,true
13,205,Gaming Console,Electronics,2024-02-05,null,true
14,206,Smartwatch,Electronics,2024-02-06,null,true
15,206,Smartwatch,Electronics,2024-02-06,null,true


## SCD TYPE-3

In [0]:
CREATE OR REPLACE TABLE sales_scd.DimProducts_SCD3
(
    ProductID INT,
    CurrentProductName STRING,
    PreviousProductName STRING,
    ProductCategory STRING
);

In [0]:
INSERT INTO sales_scd.DimProducts_SCD3
SELECT DISTINCT
    ProductID,
    ProductName AS CurrentProductName,
    NULL AS PreviousProductName,
    ProductCategory
FROM sales_scd.orders
WHERE OrderDate <= '2024-02-10';

num_affected_rows,num_inserted_rows
10,10


In [0]:
UPDATE sales_scd.DimProducts_SCD3
SET
    PreviousProductName = CurrentProductName,
    CurrentProductName = 'Gaming Laptop'
WHERE ProductID = 201;

num_affected_rows
2
